# Tool Calling

# Define a tool function

In [1]:
from langchain.tools import tool


# LangChain structured tool 
@tool
def get_order_status(order_id: str) -> dict:
    """
    Retrieves the status of a customer's order based on the provided order ID.

    Args:
        order_id (str): The unique identifier for the customer's order.
    
    Returns:
        dict: A dictionary containing the order status and estimated delivery date.
    """

    # Mocked order status data for demonstration purposes
    orders = {
        "A1024": {
            "status": "Shipped",
            "estimated_delivery": "2023-07-25"
        },
        "B2048": {
            "status": "Processing",
            "estimated_delivery": "2023-07-30"
        },
        "C4096": {
            "status": "Delivered",
            "estimated_delivery": "2023-07-20"   
        }
    }

    status = orders.get(order_id, {"status": "Order ID not found", "estimated_delivery": None })
    return status

In [2]:
# Create a chat model instance and bind the tool function to it
from langchain.chat_models import init_chat_model 

model = init_chat_model("gpt-5-nano")
tools = [get_order_status]
model_with_tools = model.bind_tools(tools)


In [3]:
# Write the system prompt to instruct the model to use the tool function
system_prompt = """
You are an e-commerce customer service assistant.

Your responsibilities:
- Help customers with questions about their orders.
- When a customer asks about an order's current status, use the
  `get_order_status` tool.
- Extract the order ID from the customer's message and pass it to the
  `order_id` argument exactly as provided.
- If the customer does not provide an order ID, ask for it before calling
  the tool.
- Base your answer on the tool result. Do not guess or fabricate an order
  status.
- If the tool returns the status "Order ID not found", tell the customer that the
  order ID could not be found and ask them to verify it.
- Do not call the tool for questions unrelated to order status.
- Respond politely and concisely in the same language used by the customer.
"""

# Tool Execution Loop

In [4]:
from langchain.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content="我的訂單 A1024 和 B2048 現在送到哪裡了？什麼時候會到？"),
]

In [5]:
# Step 1: The model generates a tool-call request.
ai_message = model_with_tools.invoke(messages)

In [6]:
# Must append the AI message to the messages list so that the model can see its own tool call request in the next step.
# If you don't append the AI message, the model will not see its own tool call request and might generate a different tool call request in the next step.
messages.append(ai_message)

In [7]:
from pprint import pprint

pprint(ai_message)


AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 317, 'prompt_tokens': 366, 'total_tokens': 683, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E5qEuu0ly884ZY1R8eXqMQcC6rAMr', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f9def-a01c-71b2-a4d4-4cbe116f1b40-0', tool_calls=[{'name': 'get_order_status', 'args': {'order_id': 'A1024'}, 'id': 'call_49RlqEqcFziXd6OSOD6VBknx', 'type': 'tool_call'}, {'name': 'get_order_status', 'args': {'order_id': 'B2048'}, 'id': 'call_PoBugMWQhr1Mx76tMi65fRku', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 366, 'output_token

In [8]:
pprint(ai_message.tool_calls)

[{'args': {'order_id': 'A1024'},
  'id': 'call_49RlqEqcFziXd6OSOD6VBknx',
  'name': 'get_order_status',
  'type': 'tool_call'},
 {'args': {'order_id': 'B2048'},
  'id': 'call_PoBugMWQhr1Mx76tMi65fRku',
  'name': 'get_order_status',
  'type': 'tool_call'}]


In [9]:
# Iterate the tool_calls list and invoke each tool function with the tool call request.
tools_by_name = {tool.name: tool for tool in tools}
print(tools_by_name)


{'get_order_status': StructuredTool(name='get_order_status', description="Retrieves the status of a customer's order based on the provided order ID.\n\nArgs:\n    order_id (str): The unique identifier for the customer's order.\n\nReturns:\n    dict: A dictionary containing the order status and estimated delivery date.", args_schema=<class 'langchain_core.utils.pydantic.get_order_status'>, func=<function get_order_status at 0x103f854e0>)}


In [10]:
for tool_call in ai_message.tool_calls:
    selected_tool = tools_by_name.get(tool_call.get("name"))
    if selected_tool:
        tool_message = selected_tool.invoke(tool_call)
        messages.append(tool_message)


In [11]:

pprint(messages)

[SystemMessage(content='\nYou are an e-commerce customer service assistant.\n\nYour responsibilities:\n- Help customers with questions about their orders.\n- When a customer asks about an order\'s current status, use the\n  `get_order_status` tool.\n- Extract the order ID from the customer\'s message and pass it to the\n  `order_id` argument exactly as provided.\n- If the customer does not provide an order ID, ask for it before calling\n  the tool.\n- Base your answer on the tool result. Do not guess or fabricate an order\n  status.\n- If the tool returns the status "Order ID not found", tell the customer that the\n  order ID could not be found and ask them to verify it.\n- Do not call the tool for questions unrelated to order status.\n- Respond politely and concisely in the same language used by the customer.\n', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='我的訂單 A1024 和 B2048 現在送到哪裡了？什麼時候會到？', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', a

In [12]:
# Step 3: The model uses the tool result to generate the final response.
final_response = model_with_tools.invoke(messages)


In [13]:

pprint(final_response.content)

('以下是兩筆訂單的最新狀態：\n'
 '\n'
 '- 訂單 A1024：已出貨，預計送達日期 2023-07-25。\n'
 '- 訂單 B2048：處理中，預計送達日期 2023-07-30。\n'
 '\n'
 '系統目前只顯示狀態與預計到貨日，未提供具體的配送地址或實時定位。如果你需要更精確的追蹤資訊（如目前位置或追蹤號），告訴我，我可以協助你進一步查詢或提供下一步的支援選項。')


## Agent


In [14]:
from langchain.agents import create_agent

model_name ="gpt-5-nano"

# system_prompt: 使用先前定義的 system_prompt

tools = [get_order_status]

agent = create_agent(
    model=model_name,
    tools=tools,
    system_prompt=system_prompt
)

In [15]:
human_message = "我的訂單 A1024 及 B2048 現在送到哪裡了？什麼時候會到？"

response = agent.invoke({"messages": [{"role": "human", "content": human_message}]})


In [16]:

final_answer = response["messages"][-1].content
print(final_answer)

以下是兩筆訂單的最新狀態（根據系統回覆）：

- 訂單 A1024
  - 狀態：已出貨（Shipped）
  - 預計送達日期：2023-07-25

- 訂單 B2048
  - 狀態：處理中（Processing）
  - 預計送達日期：2023-07-30

系統目前未提供具體的送貨地址或實時位置。如需要，我可以再為你查詢最新的追蹤資訊或設定到貨通知。需要我幫你追蹤或提供其他協助嗎？
